In [2]:
import torch
from module.model_build import *

device = "cuda" if torch.cuda.is_available() else 'cpu'
print(device)
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
print(device)





cuda
cuda


In [5]:
FORM_VOCAB_SIZE = 30000
TAG_VOCAB_SIZE = 60
DEC_VOCAB_SIZE = 30000
MAX_LEN = 150
EMBED_DIM = 256
N_HEADS = 8
N_LAYERS = 6
DROPOUT = 0.1
FFN_DIM = 512
FFN_DIM = 512

enc_embedding = EncoderEmbeddingLayer(form_vocab_size=FORM_VOCAB_SIZE, tag_vocab_size=TAG_VOCAB_SIZE, max_len=MAX_LEN, embed_dim=EMBED_DIM, dropout=DROPOUT)
dec_embedding = DecoderEmbeddingLayer(vocab_size=DEC_VOCAB_SIZE, max_len=MAX_LEN, embed_dim=EMBED_DIM, dropout=DROPOUT)

encoder = Encoder(embed_dim=EMBED_DIM, n_layers=N_LAYERS, n_heads=N_HEADS, ffn_dim=FFN_DIM, dropout=DROPOUT, max_len=MAX_LEN)
decoder = Decoder(embed_dim=EMBED_DIM, output_dim=FORM_VOCAB_SIZE, n_layers=N_LAYERS, n_heads=N_HEADS, ffn_dim=FFN_DIM, dropout=DROPOUT, embedding=dec_embedding, max_len=MAX_LEN)

transformer = Transformer(encoder=encoder, decoder=decoder, enc_embedding=enc_embedding, dec_embedding=dec_embedding)
transformer = transformer.to(device)

In [9]:
checkpoint = torch.load("checkpoint/final_model.pt", map_location=device)
print(checkpoint.keys())
model_to_load = transformer.module if hasattr(transformer, "module") else transformer

model_to_load.load_state_dict(checkpoint)

odict_keys(['encoder.encoder.0.attn_layer_norm.weight', 'encoder.encoder.0.attn_layer_norm.bias', 'encoder.encoder.0.ffn_layer_norm.weight', 'encoder.encoder.0.ffn_layer_norm.bias', 'encoder.encoder.0.self_attention.causal_mask', 'encoder.encoder.0.self_attention.w_q.weight', 'encoder.encoder.0.self_attention.w_q.bias', 'encoder.encoder.0.self_attention.w_k.weight', 'encoder.encoder.0.self_attention.w_k.bias', 'encoder.encoder.0.self_attention.w_v.weight', 'encoder.encoder.0.self_attention.w_v.bias', 'encoder.encoder.0.self_attention.w_o.weight', 'encoder.encoder.0.self_attention.w_o.bias', 'encoder.encoder.0.ffn_layer.ffn.0.weight', 'encoder.encoder.0.ffn_layer.ffn.0.bias', 'encoder.encoder.0.ffn_layer.ffn.3.weight', 'encoder.encoder.0.ffn_layer.ffn.3.bias', 'encoder.encoder.1.attn_layer_norm.weight', 'encoder.encoder.1.attn_layer_norm.bias', 'encoder.encoder.1.ffn_layer_norm.weight', 'encoder.encoder.1.ffn_layer_norm.bias', 'encoder.encoder.1.self_attention.causal_mask', 'encoder.enc

<All keys matched successfully>

In [ ]:
"""
checkpoint = torch.load("checkpoint_epoch_3.pt", map_location=device)
model_to_load = model.module if hasattr(model, "module") else model
model_to_load.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
start_epoch = checkpoint["epoch"] + 1
"""

In [11]:
from tokenizers import Tokenizer
import json

form_tokenizer = Tokenizer.from_file("../workspace/form_bpe_tokenizer.json")
en_tokenizer = Tokenizer.from_file("../workspace/en_bpe_tokenizer.json")
with open("../workspace/ko_tag_vocab.json", 'r', encoding='utf-8') as f:
    tag_tokenizer = json.load(f)
with open("../workspace/ko_tag_vocab_reverse.json", 'r', encoding='utf-8') as f:
    tag_tokenizer_r = json.load(f)

from kiwipiepy import Kiwi
kiwi = Kiwi()




In [32]:
token = en_tokenizer.encode("I love you").ids
print(token)
print(type(token))
print(en_tokenizer.decode(token))

[262, 3331, 342]
<class 'list'>
 I love you


In [44]:
def set_ft(input):
    form_tokens = []
    tag_tokens = []
    for token in kiwi.tokenize(input):
        form_tokens.append(token.form)
        tag_tokens.append(token.tag)
    form_tokens = [form_tokenizer.encode(t).ids for t in form_tokens]
    tag_tokens = [tag_tokenizer[t] for t in tag_tokens]
    
    form_t = []
    tag_t = []
    for i in range(len(form_tokens)):
       count = len(form_tokens[i])
       form_t.extend(form_tokens[i])
       tag_t.extend([tag_tokens[i]] * count)
    
    return form_t, tag_t 

def use_model(model, input):
    form_tokens, tag_tokens = set_ft(input)
    form_tokens, tag_tokens = torch.tensor(form_tokens, device=device).unsqueeze(0), torch.tensor(tag_tokens, device=device).unsqueeze(0)
    en_tokens = torch.tensor([2], device=device).unsqueeze(0)
    
    model.eval()
    stop = 0
    
    with torch.no_grad():
        while stop != 3:
            logits, _ = transformer(form_tokens, tag_tokens, en_tokens)
            next_logit = logits[:, -1, :]
            next_token = torch.argmax(next_logit, dim=-1)
            stop = next_token[0].item()
            en_tokens = torch.concat((en_tokens, next_token.unsqueeze(0)), dim=-1)
            
    output = en_tokenizer.decode(en_tokens.squeeze(0).tolist())
    return output

In [52]:
print(use_model(model=transformer, input="한영 번역기 완성했습니다."))

 I finished a friend with a friend.
